# Lectura de paquetes y data

In [1]:
import warnings
import os
import time
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from statsmodels.tsa.seasonal import STL

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.neighbors import KNeighborsRegressor

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error, mean_squared_error

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)  # Reduce verbosity de Optuna


# Agrega todo el directorio padre al path
sys.path.append(os.path.abspath(".."))
from src.utils_ml import ml_training_utils as ml_utils
from src.utils_ml import ml_feature_engineering as fe_utils
from src.utils_ml import ml_plotting as plot_utils

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

/opt/homebrew/Caskroom/miniconda/base/envs/maestria/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Utilizamos paths relativos para la lectura de la data

In [2]:
filename = "ml_pipeline_p_sku.ipynb"  # nombre del archivo actual
print(f"Current absolute path: {os.getcwd()}\n")

# Especificamos la ruta del directorio actual y los directorios de datos y salida
ACTUAL_DIR = os.path.dirname(os.path.abspath(filename))
BASE_DIR = os.path.dirname(ACTUAL_DIR)
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "output")

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

Current absolute path: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi/notebooks

BASE_DIR: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi
DATA_DIR: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi/data
OUTPUT_DIR: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi/data/output


In [3]:
# Cargar el archivo de Excel
file_path = os.path.join(DATA_DIR, "data_demanda.xlsx")
df_base = pd.read_excel(file_path, sheet_name="data")
df_base = df_base.drop("Cliente", axis=1)

df_base.shape

(6834, 11)

In [4]:
df_base.head(5)

,Fe.prefer.entrega,Day_of_the_Week,Pedidos,Sell_In,Sell_in_on_time,CEDIS,SKU,Mes_Año,Sell_in_rezago,porcentaje_on_time,Subcategoria
0,2024-07-05,Friday,16200,16200,16200,7,SKU7,72024,0,1.0,BEBIDAS DE YOGUR
1,2024-07-05,Friday,19800,19800,19800,6,SKU5,72024,0,1.0,BEBIDAS DE YOGUR
2,2024-07-05,Friday,4950,4950,4950,4,SKU6,72024,0,1.0,BEBIDAS DE YOGUR
3,2024-07-05,Friday,8100,8100,8100,6,SKU4,72024,0,1.0,BEBIDAS DE YOGUR
4,2024-07-05,Friday,18600,18600,18600,7,SKU3,72024,0,1.0,BEBIDAS DE YOGUR


In [5]:
# Filtrar los datos relevantes para este analisis

df = (
    df_base[["Fe.prefer.entrega", "SKU", "Pedidos"]]
    .copy()
    .rename(
        columns={
            "Fe.prefer.entrega": "Fecha",
        }
    )
)
df["Pedidos"] = pd.to_numeric(df["Pedidos"], errors="coerce")

In [6]:
df

,Fecha,SKU,Pedidos
0,2024-07-05,SKU7,16200
1,2024-07-05,SKU5,19800
2,2024-07-05,SKU6,4950
3,2024-07-05,SKU4,8100
4,2024-07-05,SKU3,18600
...,...,...,...
6829,2025-04-10,SKU14,1560
6830,2025-04-10,SKU16,3900
6831,2025-04-10,SKU8,4200
6832,2025-04-10,SKU22,2160


# Preparación de la data

In [7]:
### Primero, nos aseguramos de que se cuente un dato por SKU por dia
# -------

# rango completo de fechas desde la más antigua hasta la más reciente
fecha_min = df["Fecha"].min()
fecha_max = df["Fecha"].max()
rango_fechas = pd.date_range(start=fecha_min, end=fecha_max, freq="D")

# Obtenemos todos los SKUs únicos
skus = df["SKU"].unique()

# DataFrame con todas las combinaciones de SKU y fecha
combinaciones_completas = pd.MultiIndex.from_product(
    [rango_fechas, skus], names=["Fecha", "SKU"]
).to_frame(index=False)

# Unir con el dataframe original para rellenar con ceros donde falten datos
df_completo = combinaciones_completas.merge(df, on=["Fecha", "SKU"], how="left")

# Rellenar valores faltantes de pedidos con 0
df_completo["Pedidos"] = df_completo["Pedidos"].fillna(0).astype(int)

# Ordenar por SKU y Fecha (opcional)
df_completo = df_completo.sort_values(["SKU", "Fecha"]).reset_index(drop=True)

df = df_completo.copy()

In [8]:
# Modificar nombre de columnas
df.columns = df.columns.str.replace(".", "_", regex=False).str.lower()

In [9]:
df.shape

(7280, 3)

# EDA

## general

In [10]:
df.isna().sum()

fecha      0
sku        0
pedidos    0
dtype: int64

In [11]:
# porcentaje de ceros por sku
porcentaje_ceros = (
    df.groupby("sku")["pedidos"]
    .apply(lambda x: (x == 0).mean() * 100)
    .reset_index(name="prct_ceros")
    .round(2)
)

# promedio, mediana y desviacion estandar por sku excluyendo ceros
df_temp = df[df["pedidos"] > 0].copy()
promedio = df_temp.groupby("sku")["pedidos"].mean().reset_index(name="Promedio").round()
mediana = df_temp.groupby("sku")["pedidos"].median().reset_index(name="Mediana").round()
desviacion = (
    df_temp.groupby("sku")["pedidos"].std().reset_index(name="Desviacion").round()
)
maximo = df_temp.groupby("sku")["pedidos"].max().reset_index(name="Maximo").round()

# Porcentaje de valores outliers por SKU excluyendo ceros
porcentaje_outliers = (
    df_temp.groupby("sku")["pedidos"]
    .apply(fe_utils.calcular_outliers_porcentaje)
    .reset_index(name="prct_outliers")
    .round(2)
)

# Unir las tablas
tabla_total = pd.merge(porcentaje_ceros, porcentaje_outliers, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, promedio, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, mediana, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, desviacion, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, maximo, on="sku", how="outer")
tabla_total.sort_values(by="prct_ceros", ascending=False)

,sku,prct_ceros,prct_outliers,Promedio,Mediana,Desviacion,Maximo
6,SKU15,16.79,7.30,1004.0,840.0,647.0,3960
7,SKU16,14.29,2.08,4624.0,4688.0,1172.0,7500
0,SKU1,12.50,1.63,17878.0,15000.0,13859.0,84600
24,SKU8,11.79,5.26,1067.0,900.0,893.0,5160
4,SKU13,11.07,4.42,502.0,420.0,285.0,1500
3,SKU12,10.36,3.59,460.0,420.0,283.0,1800
8,SKU17,7.86,2.71,11496.0,9030.0,7494.0,52200
25,SKU9,7.86,8.53,799.0,600.0,641.0,6000
5,SKU14,7.86,2.71,2706.0,1920.0,2088.0,15240
10,SKU19,6.07,3.42,3103.0,2640.0,1959.0,10800


## Tendencias

In [12]:
sku = "SKU5"
print(f"Analizando el SKU: {sku}")

Analizando el SKU: SKU5


In [13]:
df_sku = df[df["sku"] == sku].copy()
df_sku = df_sku.drop("sku", axis=1)

# graficamos la serie de tiempo del SKU seleccionado usando plotly
fig = px.line(
    df_sku,
    x="fecha",
    y="pedidos",
    title=f"Serie de tiempo de Pedidos para {sku}:",
)
fig.update_layout(
    xaxis_title="Fecha",
)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="LightGray")
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="LightGray")
fig.update_traces(line=dict(color="blue", width=2))
fig.show()


### Descomposición STL

In [14]:
# graficamos la tendencia y estacionalidad de cada SKU usando STL
plot_utils.graficar_serie_con_descomposicion(df_sku, sku=sku, periodo=7)


# Feature engineering

## Variables temporales

In [15]:
df = fe_utils.create_temporal_features(df, "fecha")
df.shape, df.columns

((7280, 10),
 Index(['fecha', 'sku', 'pedidos', 'day_of_week', 'day_of_month', 'is_weekend',
        'week_of_month', 'is_start_of_month', 'is_near_quincena',
        'is_vacation'],
       dtype='object'))

## Variables tipo lag

In [16]:
df = fe_utils.create_lag_features(
    df, "pedidos", "sku", "fecha", max_daily_lag=14, weekday_lags=3
)
df.shape, df.columns

((7280, 27),
 Index(['fecha', 'sku', 'pedidos', 'day_of_week', 'day_of_month', 'is_weekend',
        'week_of_month', 'is_start_of_month', 'is_near_quincena', 'is_vacation',
        'pedidos_lag_1', 'pedidos_lag_2', 'pedidos_lag_3', 'pedidos_lag_4',
        'pedidos_lag_5', 'pedidos_lag_6', 'pedidos_lag_7', 'pedidos_lag_8',
        'pedidos_lag_9', 'pedidos_lag_10', 'pedidos_lag_11', 'pedidos_lag_12',
        'pedidos_lag_13', 'pedidos_lag_14', 'pedidos_weekday_lag_1',
        'pedidos_weekday_lag_2', 'pedidos_weekday_lag_3'],
       dtype='object'))

## Variables tipo promedio moviles

In [17]:
df = fe_utils.create_rolling_features(df, "pedidos", "sku", "fecha")
df.shape, df.columns

((7280, 32),
 Index(['fecha', 'sku', 'pedidos', 'day_of_week', 'day_of_month', 'is_weekend',
        'week_of_month', 'is_start_of_month', 'is_near_quincena', 'is_vacation',
        'pedidos_lag_1', 'pedidos_lag_2', 'pedidos_lag_3', 'pedidos_lag_4',
        'pedidos_lag_5', 'pedidos_lag_6', 'pedidos_lag_7', 'pedidos_lag_8',
        'pedidos_lag_9', 'pedidos_lag_10', 'pedidos_lag_11', 'pedidos_lag_12',
        'pedidos_lag_13', 'pedidos_lag_14', 'pedidos_weekday_lag_1',
        'pedidos_weekday_lag_2', 'pedidos_weekday_lag_3', 'pedidos_rolling_2',
        'pedidos_rolling_7', 'pedidos_prev_week_avg', 'pedidos_dow_avg_2wks',
        'pedidos_dow_avg_3wks'],
       dtype='object'))

## Variables lags de STL

In [18]:
df = fe_utils.create_stl_features(
    df, "pedidos", "sku", "fecha", seasonal=7, stl_lags=14
)
df.shape, df.columns


((7280, 60),
 Index(['fecha', 'sku', 'pedidos', 'day_of_week', 'day_of_month', 'is_weekend',
        'week_of_month', 'is_start_of_month', 'is_near_quincena', 'is_vacation',
        'pedidos_lag_1', 'pedidos_lag_2', 'pedidos_lag_3', 'pedidos_lag_4',
        'pedidos_lag_5', 'pedidos_lag_6', 'pedidos_lag_7', 'pedidos_lag_8',
        'pedidos_lag_9', 'pedidos_lag_10', 'pedidos_lag_11', 'pedidos_lag_12',
        'pedidos_lag_13', 'pedidos_lag_14', 'pedidos_weekday_lag_1',
        'pedidos_weekday_lag_2', 'pedidos_weekday_lag_3', 'pedidos_rolling_2',
        'pedidos_rolling_7', 'pedidos_prev_week_avg', 'pedidos_dow_avg_2wks',
        'pedidos_dow_avg_3wks', 'stl_trend_lag_1', 'stl_trend_lag_2',
        'stl_trend_lag_3', 'stl_trend_lag_4', 'stl_trend_lag_5',
        'stl_trend_lag_6', 'stl_trend_lag_7', 'stl_trend_lag_8',
        'stl_trend_lag_9', 'stl_trend_lag_10', 'stl_trend_lag_11',
        'stl_trend_lag_12', 'stl_trend_lag_13', 'stl_trend_lag_14',
        'stl_seasonal_lag_1', 'stl

## Aplanamiento de outliers en demanda

In [19]:
df = fe_utils.cap_upper_outliers(df, "pedidos", "sku")

## Ajustes finales a la data

In [20]:
# Eliminamos todas las filas con valores NaN para que no afecten el entrenamiento
df.dropna(inplace=True)
df.shape, df.columns

((6543, 60),
 Index(['fecha', 'sku', 'pedidos', 'day_of_week', 'day_of_month', 'is_weekend',
        'week_of_month', 'is_start_of_month', 'is_near_quincena', 'is_vacation',
        'pedidos_lag_1', 'pedidos_lag_2', 'pedidos_lag_3', 'pedidos_lag_4',
        'pedidos_lag_5', 'pedidos_lag_6', 'pedidos_lag_7', 'pedidos_lag_8',
        'pedidos_lag_9', 'pedidos_lag_10', 'pedidos_lag_11', 'pedidos_lag_12',
        'pedidos_lag_13', 'pedidos_lag_14', 'pedidos_weekday_lag_1',
        'pedidos_weekday_lag_2', 'pedidos_weekday_lag_3', 'pedidos_rolling_2',
        'pedidos_rolling_7', 'pedidos_prev_week_avg', 'pedidos_dow_avg_2wks',
        'pedidos_dow_avg_3wks', 'stl_trend_lag_1', 'stl_trend_lag_2',
        'stl_trend_lag_3', 'stl_trend_lag_4', 'stl_trend_lag_5',
        'stl_trend_lag_6', 'stl_trend_lag_7', 'stl_trend_lag_8',
        'stl_trend_lag_9', 'stl_trend_lag_10', 'stl_trend_lag_11',
        'stl_trend_lag_12', 'stl_trend_lag_13', 'stl_trend_lag_14',
        'stl_seasonal_lag_1', 'stl

In [21]:
df

,fecha,sku,pedidos,day_of_week,day_of_month,is_weekend,week_of_month,is_start_of_month,is_near_quincena,is_vacation,pedidos_lag_1,pedidos_lag_2,pedidos_lag_3,pedidos_lag_4,pedidos_lag_5,pedidos_lag_6,pedidos_lag_7,pedidos_lag_8,pedidos_lag_9,pedidos_lag_10,pedidos_lag_11,pedidos_lag_12,pedidos_lag_13,pedidos_lag_14,pedidos_weekday_lag_1,pedidos_weekday_lag_2,pedidos_weekday_lag_3,pedidos_rolling_2,pedidos_rolling_7,pedidos_prev_week_avg,pedidos_dow_avg_2wks,pedidos_dow_avg_3wks,stl_trend_lag_1,stl_trend_lag_2,stl_trend_lag_3,stl_trend_lag_4,stl_trend_lag_5,stl_trend_lag_6,stl_trend_lag_7,stl_trend_lag_8,stl_trend_lag_9,stl_trend_lag_10,stl_trend_lag_11,stl_trend_lag_12,stl_trend_lag_13,stl_trend_lag_14,stl_seasonal_lag_1,stl_seasonal_lag_2,stl_seasonal_lag_3,stl_seasonal_lag_4,stl_seasonal_lag_5,stl_seasonal_lag_6,stl_seasonal_lag_7,stl_seasonal_lag_8,stl_seasonal_lag_9,stl_seasonal_lag_10,stl_seasonal_lag_11,stl_seasonal_lag_12,stl_seasonal_lag_13,stl_seasonal_lag_14
21,2024-07-26,SKU1,0.0,4,26,0,4,0,0,1,30000.0,54000.0,6000.0,6000.0,6000.0,3000.0,6000.0,30000.0,36000.0,2400.0,2400.0,2400.0,0.0,0.0,6000.0,0.0,7200.0,42000.0,15857.142857,13200.000000,6000.0,6600.0,11912.075530,12067.577030,12245.869160,12505.547013,12750.034828,12822.162603,12637.629174,12266.527681,11860.325068,11553.859386,11318.073301,11162.328801,11142.505288,11307.379638,18847.399955,21172.839407,-7645.745853,-7408.951414,-6810.414165,-9948.694610,-9241.110114,23936.177946,17935.030019,-8482.941590,-8702.020679,-7241.913414,-9376.524091,-9032.297251
22,2024-07-27,SKU1,0.0,5,27,1,4,0,0,1,0.0,30000.0,54000.0,6000.0,6000.0,6000.0,3000.0,6000.0,30000.0,36000.0,2400.0,2400.0,2400.0,0.0,3000.0,0.0,7200.0,30000.0,17500.000000,11742.857143,3000.0,5100.0,11781.079855,11912.075530,12067.577030,12245.869160,12505.547013,12750.034828,12822.162603,12637.629174,12266.527681,11860.325068,11553.859386,11318.073301,11162.328801,11142.505288,-9310.776185,18847.399955,21172.839407,-7645.745853,-7408.951414,-6810.414165,-9948.694610,-9241.110114,23936.177946,17935.030019,-8482.941590,-8702.020679,-7241.913414,-9376.524091
24,2024-07-29,SKU1,6000.0,0,29,0,5,0,1,1,6000.0,0.0,0.0,30000.0,54000.0,6000.0,6000.0,6000.0,3000.0,6000.0,30000.0,36000.0,2400.0,2400.0,6000.0,2400.0,3000.0,6000.0,20400.000000,12771.428571,4200.0,3800.0,11759.561758,11686.683023,11781.079855,11912.075530,12067.577030,12245.869160,12505.547013,12750.034828,12822.162603,12637.629174,12266.527681,11860.325068,11553.859386,11318.073301,-6275.247195,-10356.559802,-9310.776185,18847.399955,21172.839407,-7645.745853,-7408.951414,-6810.414165,-9948.694610,-9241.110114,23936.177946,17935.030019,-8482.941590,-8702.020679
25,2024-07-30,SKU1,6000.0,1,30,0,5,0,1,1,6000.0,6000.0,0.0,0.0,30000.0,54000.0,6000.0,6000.0,6000.0,3000.0,6000.0,30000.0,36000.0,2400.0,6000.0,2400.0,3000.0,6000.0,20400.000000,13285.714286,4200.0,3800.0,12147.310782,11759.561758,11686.683023,11781.079855,11912.075530,12067.577030,12245.869160,12505.547013,12750.034828,12822.162603,12637.629174,12266.527681,11860.325068,11553.859386,-6213.341647,-6275.247195,-10356.559802,-9310.776185,18847.399955,21172.839407,-7645.745853,-7408.951414,-6810.414165,-9948.694610,-9241.110114,23936.177946,17935.030019,-8482.941590
26,2024-07-31,SKU1,36000.0,2,31,0,5,0,1,1,6000.0,6000.0,6000.0,0.0,0.0,30000.0,54000.0,6000.0,6000.0,6000.0,3000.0,6000.0,30000.0,36000.0,54000.0,36000.0,24000.0,6000.0,20400.000000,15857.142857,45000.0,38000.0,12733.007654,12147.310782,11759.561758,11686.683023,11781.079855,11912.075530,12067.577030,12245.869160,12505.547013,12750.034828,12822.162603,12637.629174,12266.527681,11860.325068,-6884.697557,-6213.341647,-6275.247195,-10356.559802,-9310.776185,18847.399955,21172.839407,-7645.745853,-7408.951414,-6810.414165,-9948.694610,-9241.110114,23936.177946,17935.030019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

# Modelling 

In [22]:
## Preparación de data para modelling

# Separamos el conjunto de datos en entrenamiento y prueba, usando los últimos 7 días como prueba
test = df.tail(7)
df_mod = df[:-7].copy()

# Identificamos las variables predictoras
features = df_mod.columns.difference(["fecha", "sku", "pedidos"]).to_list()

# definimos el numero de ventanas de evaluación y el tamaño de las ventanas
val_iter = 3
val_size = 7

## Evaluación modelos XGBoost

In [23]:
# Creamos un DataFrame para almacenar los resultados de cada SKU
# Este DataFrame contendrá el SKU, los parámetros del modelo y las métricas de evaluación
##########

results = []

for sku in df_mod["sku"].unique():
    print("\n-----------------------")
    print(f"\n🔍 Optimizando para SKU: {sku}\n")

    # Filtramos el DataFrame para el SKU actual
    df_sku = df_mod[df_mod["sku"] == sku].copy()

    # Calculamos el tamaño del conjunto de entrenamiento
    train_size = df_sku.shape[0] - val_iter * val_size

    # Definimos el espacio de búsqueda de hiperparámetros para el modelo XGBoost
    # Usamos funciones lambda para que Optuna pueda sugerir valores
    param_grid = {
        "max_depth": lambda trial: trial.suggest_int("max_depth", 1, 5, step=1),
        "learning_rate": lambda trial: trial.suggest_float(
            "learning_rate", 0.001, 0.1, step=0.001
        ),
        "n_estimators": lambda trial: trial.suggest_int(
            "n_estimators", 50, 3000, step=50
        ),
        "subsample": lambda trial: trial.suggest_float(
            "subsample", 0.5, 1.0, step=0.02
        ),
        "colsample_bytree": lambda trial: trial.suggest_float(
            "colsample_bytree", 0.5, 1.0, step=0.02
        ),
        "gamma": lambda trial: trial.suggest_float("gamma", 1, 30, step=0.5),
        "reg_alpha": lambda trial: trial.suggest_float("reg_alpha", 1, 30, step=0.5),
        "reg_lambda": lambda trial: trial.suggest_float("reg_lambda", 1, 30, step=0.5),
        "min_child_weight": lambda trial: trial.suggest_int(
            "min_child_weight", 5, 20, step=1
        ),
        "random_state": lambda trial: 100,  # Fijamos la semilla para reproducibilidad
    }

    # Optimizamos el modelo XGBoost usando Optuna con ventana recursiva
    study = ml_utils.optimize_model_with_optuna(
        model_class=XGBRegressor,
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=125,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    # Obtenemos el mejor trial del estudio
    # y almacenamos los resultados en un diccionario
    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "XGBRegressor",
            "n_trials": len(study.trials),
        }
    )

    print("\n-----------------------")


df_results_xgboost = pd.DataFrame(results)


-----------------------

🔍 Optimizando para SKU: SKU1



  0%|          | 0/125 [00:00<?, ?it/s]

100%|██████████| 125/125 [10:36<00:00,  5.10s/it]



📌 Mejores Trials:
Trial 0 - SMAPE: 27.4600, GAP: 1.8300
Trial 1 - SMAPE: 27.4400, GAP: 3.9200
Trial 2 - SMAPE: 29.0000, GAP: 1.7000
Trial 3 - SMAPE: 27.0400, GAP: 5.3700
Trial 4 - SMAPE: 26.7900, GAP: 6.5500

Número total de Best Trials ('Frente de pareto'): 5

🏆 Best Trial Params:
{'max_depth': 2, 'learning_rate': 0.021, 'n_estimators': 2800, 'subsample': 0.9, 'colsample_bytree': 0.98, 'gamma': 2.0, 'reg_alpha': 5.0, 'reg_lambda': 6.5, 'min_child_weight': 6}

-----------------------

-----------------------

🔍 Optimizando para SKU: SKU10



100%|██████████| 125/125 [05:52<00:00,  2.82s/it]


...


📌 Mejores Trials:
Trial 0 - SMAPE: 47.1100, GAP: 8.0500
Trial 1 - SMAPE: 48.1800, GAP: 7.4900
Trial 2 - SMAPE: 48.4300, GAP: 6.7000
Trial 3 - SMAPE: 41.0500, GAP: 10.5100
Trial 4 - SMAPE: 46.9900, GAP: 8.4300

Número total de Best Trials ('Frente de pareto'): 10

🏆 Best Trial Params:
{'max_depth': 1, 'learning_rate': 0.002, 'n_estimators': 900, 'subsample': 0.98, 'colsample_bytree': 0.78, 'gamma': 19.0, 'reg_alpha': 1.0, 'reg_lambda': 17.5, 'min_child_weight': 5}

-----------------------

-----------------------

🔍 Optimizando para SKU: SKU8



100%|██████████| 125/125 [04:07<00:00,  1.98s/it]



📌 Mejores Trials:
Trial 0 - SMAPE: 86.8800, GAP: 32.1800
Trial 1 - SMAPE: 88.0000, GAP: 30.9700
Trial 2 - SMAPE: 86.6300, GAP: 33.9200
Trial 3 - SMAPE: 85.7000, GAP: 34.3600
Trial 4 - SMAPE: 85.5000, GAP: 34.3700

Número total de Best Trials ('Frente de pareto'): 16

🏆 Best Trial Params:
{'max_depth': 1, 'learning_rate': 0.005, 'n_estimators': 800, 'subsample': 0.94, 'colsample_bytree': 0.88, 'gamma': 18.0, 'reg_alpha': 23.0, 'reg_lambda': 2.5, 'min_child_weight': 19}

-----------------------

-----------------------

🔍 Optimizando para SKU: SKU9



100%|██████████| 125/125 [04:49<00:00,  2.31s/it]


📌 Mejores Trials:
Trial 0 - SMAPE: 59.7900, GAP: 41.2700
Trial 1 - SMAPE: 61.7200, GAP: 38.6600
Trial 2 - SMAPE: 62.8000, GAP: 19.1100
Trial 3 - SMAPE: 60.2200, GAP: 40.3000
Trial 4 - SMAPE: 63.1500, GAP: 18.3200

Número total de Best Trials ('Frente de pareto'): 7

🏆 Best Trial Params:
{'max_depth': 2, 'learning_rate': 0.065, 'n_estimators': 1300, 'subsample': 1.0, 'colsample_bytree': 0.86, 'gamma': 21.0, 'reg_alpha': 17.0, 'reg_lambda': 14.0, 'min_child_weight': 7}

-----------------------


In [24]:
df_results_xgboost

,sku,study,best_params,best_smape,best_gap,model,n_trials
0,SKU1,<optuna.study.study.Study object at 0x1380c0ad0>,"{'max_depth': 2, 'learning_rate': 0.021, 'n_es...",27.46,1.83,XGBRegressor,125
1,SKU10,<optuna.study.study.Study object at 0x1380634d0>,"{'max_depth': 5, 'learning_rate': 0.076, 'n_es...",25.84,3.78,XGBRegressor,125
2,SKU11,<optuna.study.study.Study object at 0x138063b10>,"{'max_depth': 1, 'learning_rate': 0.001, 'n_es...",35.77,3.86,XGBRegressor,125
3,SKU12,<optuna.study.study.Study object at 0x12fd3b490>,"{'max_depth': 3, 'learning_rate': 0.0090000000...",78.96,23.59,XGBRegressor,125
4,SKU13,<optuna.study.study.Study object at 0x1380ac8a0>,"{'max_depth': 4, 'learning_rate': 0.058, 'n_es...",46.82,13.10,XGBRegressor,125
5,SKU14,<optuna.study.study.Study object at 0x1380fa0f0>,"{'max_depth': 1, 'learning_rate': 0.063, 'n_es...",34.84,14.82,XGBRegressor,125
6,SKU15,<optuna.study.study.Study object at 0x138029040>,"{'max_depth': 3, 'learning_rate': 0.028, 'n_es...",43.36,20.57,XGBRegressor,125
7,SKU16,<optuna.study.study.Study object at 0x1380298c0>,"{'max_depth': 1, 'learning_rate': 0.0720000000...",22.39,9.38,XGBRegressor,125
8,SKU17,<optuna.study.study.Study object at 0x12fd5bb50>,"{'max_depth': 1, 'learning_rate': 0.08, 'n_est...",32.04,4.58,XGBRegressor,125
9,SKU18,<optuna.study.study.Study object at 0x139208650>,"{'max_depth': 1, 'learning_rate': 0.0530000000...",45.66,18.50,XGBRegressor,125


In [25]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_xgboost[df_results_xgboost["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Evaluación modelos Random Forest

In [26]:
# Creamos un DataFrame para almacenar los resultados de cada SKU
# Este DataFrame contendrá el SKU, los parámetros del modelo y las métricas de evaluación
##########

results = []

for sku in df_mod["sku"].unique():
    print("\n-----------------------")
    print(f"🔍 Optimizando para SKU: {sku}\n")

    # Filtramos por SKU
    df_sku = df_mod[df_mod["sku"] == sku].copy()

    # Tamaño de entrenamiento
    train_size = df_sku.shape[0] - val_iter * val_size

    # Espacio de búsqueda de hiperparámetros para Random Forest
    param_grid = {
        "n_estimators": lambda trial: trial.suggest_int(
            "n_estimators", 50, 1000, step=50
        ),
        "max_depth": lambda trial: trial.suggest_int("max_depth", 1, 10, step=1),
        "min_samples_split": lambda trial: trial.suggest_int(
            "min_samples_split", 2, 10
        ),
        "min_samples_leaf": lambda trial: trial.suggest_int(
            "min_samples_leaf", 3, 15, step=1
        ),
        "max_features": lambda trial: trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
        "bootstrap": lambda trial: trial.suggest_categorical(
            "bootstrap", [True, False]
        ),
        "random_state": lambda trial: 100,  # Fijamos el random_state para reproducibilidad
    }

    # Optimizamos usando tu función personalizada con modelo RandomForestRegressor
    study = ml_utils.optimize_model_with_optuna(
        model_class=RandomForestRegressor,
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=125,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    # Registramos los resultados
    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "RandomForestRegressor",
            "n_trials": len(study.trials),
        }
    )

    print("-----------------------")

# Creamos DataFrame con resultados
df_results_rf = pd.DataFrame(results)



-----------------------
🔍 Optimizando para SKU: SKU1



100%|██████████| 125/125 [05:10<00:00,  2.48s/it]



📌 Mejores Trials:
Trial 0 - SMAPE: 32.7100, GAP: 9.5900
Trial 1 - SMAPE: 36.5200, GAP: 6.3700
Trial 2 - SMAPE: 36.5200, GAP: 6.3700
Trial 3 - SMAPE: 35.9500, GAP: 8.3600
Trial 4 - SMAPE: 32.8700, GAP: 8.4700

Número total de Best Trials ('Frente de pareto'): 14

🏆 Best Trial Params:
{'n_estimators': 600, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': None, 'bootstrap': False}
-----------------------

-----------------------
🔍 Optimizando para SKU: SKU10



100%|██████████| 125/125 [04:54<00:00,  2.36s/it]



📌 Mejores Trials:
Trial 0 - SMAPE: 26.7000, GAP: 6.6200
Trial 1 - SMAPE: 28.5100, GAP: 1.1100
Trial 2 - SMAPE: 28.2100, GAP: 2.0600
Trial 3 - SMAPE: 27.3900, GAP: 5.7100
Trial 4 - SMAPE: 27.1700, GAP: 6.5300

Número total de Best Trials ('Frente de pareto'): 23

🏆 Best Trial Params:
{'n_estimators': 50, 'max_depth': 7, 'min_samples_split': 4, 'min_samples_leaf': 12, 'max_features': None, 'bootstrap': True}
-----------------------

-----------------------
🔍 Optimizando para SKU: SKU11



...


📌 Mejores Trials:
Trial 0 - SMAPE: 46.2300, GAP: 14.7000
Trial 1 - SMAPE: 46.5100, GAP: 13.1400
Trial 2 - SMAPE: 48.1600, GAP: 8.6900
Trial 3 - SMAPE: 45.9600, GAP: 15.5900
Trial 4 - SMAPE: 48.1700, GAP: 8.5800

Número total de Best Trials ('Frente de pareto'): 34

🏆 Best Trial Params:
{'n_estimators': 550, 'max_depth': 6, 'min_samples_split': 2, 'min_samples_leaf': 10, 'max_features': None, 'bootstrap': True}
-----------------------

-----------------------
🔍 Optimizando para SKU: SKU8



100%|██████████| 125/125 [03:09<00:00,  1.51s/it]



📌 Mejores Trials:
Trial 0 - SMAPE: 91.6000, GAP: 31.9600
Trial 1 - SMAPE: 89.5600, GAP: 36.6100
Trial 2 - SMAPE: 89.5600, GAP: 36.6100
Trial 3 - SMAPE: 88.9300, GAP: 36.7100
Trial 4 - SMAPE: 90.0000, GAP: 33.3700

Número total de Best Trials ('Frente de pareto'): 19

🏆 Best Trial Params:
{'n_estimators': 1000, 'max_depth': 1, 'min_samples_split': 10, 'min_samples_leaf': 15, 'max_features': 'sqrt', 'bootstrap': False}
-----------------------

-----------------------
🔍 Optimizando para SKU: SKU9



100%|██████████| 125/125 [05:46<00:00,  2.77s/it]


📌 Mejores Trials:
Trial 0 - SMAPE: 56.2300, GAP: 23.7800
Trial 1 - SMAPE: 57.2500, GAP: 12.4900
Trial 2 - SMAPE: 57.2500, GAP: 12.4900
Trial 3 - SMAPE: 57.1300, GAP: 13.1800
Trial 4 - SMAPE: 57.1300, GAP: 13.1800

Número total de Best Trials ('Frente de pareto'): 15

🏆 Best Trial Params:
{'n_estimators': 1000, 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'bootstrap': False}
-----------------------


In [27]:
df_results_rf

,sku,study,best_params,best_smape,best_gap,model,n_trials
0,SKU1,<optuna.study.study.Study object at 0x105e7f140>,"{'n_estimators': 600, 'max_depth': 10, 'min_sa...",32.71,9.59,RandomForestRegressor,125
1,SKU10,<optuna.study.study.Study object at 0x12fd5f4a0>,"{'n_estimators': 50, 'max_depth': 7, 'min_samp...",26.70,6.62,RandomForestRegressor,125
2,SKU11,<optuna.study.study.Study object at 0x12fd5f260>,"{'n_estimators': 550, 'max_depth': 6, 'min_sam...",34.58,5.53,RandomForestRegressor,125
3,SKU12,<optuna.study.study.Study object at 0x12fd5f380>,"{'n_estimators': 450, 'max_depth': 3, 'min_sam...",73.69,31.06,RandomForestRegressor,125
4,SKU13,<optuna.study.study.Study object at 0x12fd5f410>,"{'n_estimators': 1000, 'max_depth': 7, 'min_sa...",37.38,10.28,RandomForestRegressor,125
5,SKU14,<optuna.study.study.Study object at 0x12fd5ff50>,"{'n_estimators': 200, 'max_depth': 3, 'min_sam...",64.53,25.77,RandomForestRegressor,125
6,SKU15,<optuna.study.study.Study object at 0x12fd5f6e0>,"{'n_estimators': 600, 'max_depth': 10, 'min_sa...",50.76,14.53,RandomForestRegressor,125
7,SKU16,<optuna.study.study.Study object at 0x12fd5fad0>,"{'n_estimators': 350, 'max_depth': 8, 'min_sam...",21.51,11.82,RandomForestRegressor,125
8,SKU17,<optuna.study.study.Study object at 0x12fd5fbf0>,"{'n_estimators': 100, 'max_depth': 8, 'min_sam...",35.88,7.44,RandomForestRegressor,125
9,SKU18,<optuna.study.study.Study object at 0x12fd5fa40>,"{'n_estimators': 800, 'max_depth': 1, 'min_sam...",46.02,10.11,RandomForestRegressor,125


In [28]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_rf[df_results_rf["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Evaluación modelos Elastic Net

In [29]:
# Creamos un DataFrame para almacenar los resultados de cada SKU
# Este DataFrame contendrá el SKU, los parámetros del modelo y las métricas de evaluación
##########

results = []

for sku in df_mod["sku"].unique():
    print("\n-----------------------")
    print(f"🔍 Optimizando ElasticNet para SKU: {sku}\n")

    # Filtramos por SKU
    df_sku = df_mod[df_mod["sku"] == sku].copy()
    train_size = df_sku.shape[0] - val_iter * val_size

    # Espacio de búsqueda de hiperparámetros para ElasticNet
    param_grid = {
        "model__alpha": lambda trial: trial.suggest_float(
            "model__alpha", 0.0001, 1000.0, log=True
        ),
        "model__l1_ratio": lambda trial: trial.suggest_float(
            "model__l1_ratio", 0.0, 1.0, step=0.01
        ),  # 0 = Ridge, 1 = Lasso
        "model__random_state": lambda trial: 100,  # Fijamos el random_state para reproducibilidad
    }

    # Pipeline: escalado + modelo
    model_pipeline = Pipeline(
        [("scaler", StandardScaler()), ("model", ElasticNet(max_iter=20000))]
    )

    # Optimización con tu función
    study = ml_utils.optimize_model_with_optuna(
        model_class=lambda **params: model_pipeline.set_params(**params),
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=200,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "ElasticNet",
            "n_trials": len(study.trials),
        }
    )

    print("-----------------------")

df_results_elastic = pd.DataFrame(results)



-----------------------
🔍 Optimizando ElasticNet para SKU: SKU1



100%|██████████| 200/200 [00:28<00:00,  7.13it/s]



📌 Mejores Trials:
Trial 0 - SMAPE: 27.6900, GAP: 23.5400
Trial 1 - SMAPE: 28.0200, GAP: 23.2300
Trial 2 - SMAPE: 28.8100, GAP: 21.7800
Trial 3 - SMAPE: 27.7600, GAP: 23.4800
Trial 4 - SMAPE: 26.5600, GAP: 26.8300

Número total de Best Trials ('Frente de pareto'): 83

🏆 Best Trial Params:
{'model__alpha': 0.041858227295469716, 'model__l1_ratio': 0.96}
-----------------------

-----------------------
🔍 Optimizando ElasticNet para SKU: SKU10



100%|██████████| 200/200 [00:25<00:00,  7.83it/s]



📌 Mejores Trials:
Trial 0 - SMAPE: 42.3000, GAP: 4.5900
Trial 1 - SMAPE: 43.0100, GAP: 4.2200
Trial 2 - SMAPE: 28.1900, GAP: 5.1000
Trial 3 - SMAPE: 43.1800, GAP: 4.1300
Trial 4 - SMAPE: 42.8500, GAP: 4.3200

Número total de Best Trials ('Frente de pareto'): 25

🏆 Best Trial Params:
{'model__alpha': 148.9838559398634, 'model__l1_ratio': 0.8300000000000001}
-----------------------

-----------------------
🔍 Optimizando ElasticNet para SKU: SKU11



...


📌 Mejores Trials:
Trial 0 - SMAPE: 50.6400, GAP: 4.7500
Trial 1 - SMAPE: 50.6600, GAP: 4.6900
Trial 2 - SMAPE: 41.5200, GAP: 6.6600
Trial 3 - SMAPE: 50.6600, GAP: 4.6900
Trial 4 - SMAPE: 41.4500, GAP: 6.7600

Número total de Best Trials ('Frente de pareto'): 26

🏆 Best Trial Params:
{'model__alpha': 744.9271069389815, 'model__l1_ratio': 0.39}
-----------------------

-----------------------
🔍 Optimizando ElasticNet para SKU: SKU8



100%|██████████| 200/200 [00:09<00:00, 20.84it/s]



📌 Mejores Trials:
Trial 0 - SMAPE: 72.0200, GAP: 12.9800

Número total de Best Trials ('Frente de pareto'): 1

🏆 Best Trial Params:
{'model__alpha': 233.75241321673985, 'model__l1_ratio': 1.0}
-----------------------

-----------------------
🔍 Optimizando ElasticNet para SKU: SKU9



100%|██████████| 200/200 [00:13<00:00, 15.29it/s]


📌 Mejores Trials:
Trial 0 - SMAPE: 50.1800, GAP: 8.5800
Trial 1 - SMAPE: 50.0300, GAP: 8.7600
Trial 2 - SMAPE: 50.4600, GAP: 8.2200
Trial 3 - SMAPE: 50.6900, GAP: 7.9100
Trial 4 - SMAPE: 41.2200, GAP: 8.9400

Número total de Best Trials ('Frente de pareto'): 13

🏆 Best Trial Params:
{'model__alpha': 0.06032030579768781, 'model__l1_ratio': 0.03}
-----------------------


In [30]:
df_results_elastic

,sku,study,best_params,best_smape,best_gap,model,n_trials
0,SKU1,<optuna.study.study.Study object at 0x12fd5e8d0>,"{'model__alpha': 0.041858227295469716, 'model_...",27.69,23.54,ElasticNet,200
1,SKU10,<optuna.study.study.Study object at 0x139f962a0>,"{'model__alpha': 148.9838559398634, 'model__l1...",42.30,4.59,ElasticNet,200
2,SKU11,<optuna.study.study.Study object at 0x139f97410>,"{'model__alpha': 0.0002550264850403288, 'model...",29.89,3.42,ElasticNet,200
3,SKU12,<optuna.study.study.Study object at 0x139f96180>,"{'model__alpha': 0.009349170414488468, 'model_...",38.76,17.16,ElasticNet,200
4,SKU13,<optuna.study.study.Study object at 0x139f97770>,"{'model__alpha': 619.5023740341801, 'model__l1...",52.77,10.62,ElasticNet,200
5,SKU14,<optuna.study.study.Study object at 0x139f949e0>,"{'model__alpha': 0.00015533117206750294, 'mode...",55.22,28.78,ElasticNet,200
6,SKU15,<optuna.study.study.Study object at 0x139f97650>,"{'model__alpha': 0.041858227295469716, 'model_...",47.53,12.22,ElasticNet,200
7,SKU16,<optuna.study.study.Study object at 0x139f94710>,"{'model__alpha': 0.10558813779064824, 'model__...",20.88,14.66,ElasticNet,200
8,SKU17,<optuna.study.study.Study object at 0x139f971d0>,"{'model__alpha': 0.0012363188277052211, 'model...",26.54,21.25,ElasticNet,200
9,SKU18,<optuna.study.study.Study object at 0x139f96450>,"{'model__alpha': 64.51445379907872, 'model__l1...",50.67,15.06,ElasticNet,200


In [31]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_elastic[df_results_elastic["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Evaluación modelos KNeighbors 

In [32]:
# Creamos un DataFrame para almacenar los resultados de cada SKU
# Este DataFrame contendrá el SKU, los parámetros del modelo y las métricas de evaluación
##########

results = []

for sku in df_mod["sku"].unique():
    print("\n-----------------------")
    print(f"🔍 Optimizando KNN para SKU: {sku}\n")

    df_sku = df_mod[df_mod["sku"] == sku].copy()
    train_size = df_sku.shape[0] - val_iter * val_size

    # Pipeline: escalado obligatorio + modelo
    pipeline = Pipeline(
        [("scaler", StandardScaler()), ("model", KNeighborsRegressor())]
    )

    # Espacio de búsqueda
    param_grid = {
        "model__n_neighbors": lambda trial: trial.suggest_int(
            "model__n_neighbors", 2, 30, step=1
        ),
        "model__weights": lambda trial: trial.suggest_categorical(
            "model__weights", ["uniform", "distance"]
        ),
        "model__p": lambda trial: trial.suggest_int(
            "model__p", 1, 2
        ),  # 1 = manhattan, 2 = euclidean
        "model__leaf_size": lambda trial: trial.suggest_int(
            "model__leaf_size", 5, 100, step=1
        ),
        "model__algorithm": lambda trial: trial.suggest_categorical(
            "model__algorithm", ["auto", "ball_tree", "kd_tree"]
        ),
    }

    # Llamada a tu función personalizada
    study = ml_utils.optimize_model_with_optuna(
        model_class=lambda **params: pipeline.set_params(**params),
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=300,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "KNeighborsRegressor",
            "n_trials": len(study.trials),
        }
    )

    print("-----------------------")

df_results_knn = pd.DataFrame(results)



-----------------------
🔍 Optimizando KNN para SKU: SKU1



100%|██████████| 300/300 [00:05<00:00, 54.22it/s]



📌 Mejores Trials:
Trial 0 - SMAPE: 44.0000, GAP: 5.9000
Trial 1 - SMAPE: 36.0000, GAP: 13.8800
Trial 2 - SMAPE: 36.0000, GAP: 13.8800
Trial 3 - SMAPE: 36.0000, GAP: 13.8800
Trial 4 - SMAPE: 30.7700, GAP: 24.4100

Número total de Best Trials ('Frente de pareto'): 43

🏆 Best Trial Params:
{'model__n_neighbors': 2, 'model__weights': 'uniform', 'model__p': 2, 'model__leaf_size': 79, 'model__algorithm': 'ball_tree'}
-----------------------

-----------------------
🔍 Optimizando KNN para SKU: SKU10



100%|██████████| 300/300 [00:05<00:00, 54.97it/s]



📌 Mejores Trials:
Trial 0 - SMAPE: 33.8300, GAP: 5.3200
Trial 1 - SMAPE: 33.8300, GAP: 5.3200
Trial 2 - SMAPE: 33.8300, GAP: 5.3200
Trial 3 - SMAPE: 33.8300, GAP: 5.3200
Trial 4 - SMAPE: 32.5700, GAP: 5.4200

Número total de Best Trials ('Frente de pareto'): 55

🏆 Best Trial Params:
{'model__n_neighbors': 8, 'model__weights': 'uniform', 'model__p': 2, 'model__leaf_size': 39, 'model__algorithm': 'kd_tree'}
-----------------------

-----------------------
🔍 Optimizando KNN para SKU: SKU11



...


📌 Mejores Trials:
Trial 0 - SMAPE: 48.0900, GAP: 6.7100
Trial 1 - SMAPE: 48.0900, GAP: 6.7100
Trial 2 - SMAPE: 48.0900, GAP: 6.7100
Trial 3 - SMAPE: 48.0900, GAP: 6.7100
Trial 4 - SMAPE: 48.0900, GAP: 6.7100

Número total de Best Trials ('Frente de pareto'): 13

🏆 Best Trial Params:
{'model__n_neighbors': 29, 'model__weights': 'uniform', 'model__p': 2, 'model__leaf_size': 54, 'model__algorithm': 'auto'}
-----------------------

-----------------------
🔍 Optimizando KNN para SKU: SKU8



100%|██████████| 300/300 [00:05<00:00, 56.06it/s]



📌 Mejores Trials:
Trial 0 - SMAPE: 88.3100, GAP: 33.6900
Trial 1 - SMAPE: 88.3100, GAP: 33.6900
Trial 2 - SMAPE: 88.3100, GAP: 33.6900
Trial 3 - SMAPE: 88.3100, GAP: 33.6900
Trial 4 - SMAPE: 88.3100, GAP: 33.6900

Número total de Best Trials ('Frente de pareto'): 8

🏆 Best Trial Params:
{'model__n_neighbors': 6, 'model__weights': 'uniform', 'model__p': 2, 'model__leaf_size': 29, 'model__algorithm': 'auto'}
-----------------------

-----------------------
🔍 Optimizando KNN para SKU: SKU9



100%|██████████| 300/300 [00:05<00:00, 56.62it/s]


📌 Mejores Trials:
Trial 0 - SMAPE: 68.4200, GAP: 15.2200
Trial 1 - SMAPE: 68.4200, GAP: 15.2200
Trial 2 - SMAPE: 68.4900, GAP: 15.0700
Trial 3 - SMAPE: 68.4900, GAP: 15.0700
Trial 4 - SMAPE: 68.4900, GAP: 15.0700

Número total de Best Trials ('Frente de pareto'): 11

🏆 Best Trial Params:
{'model__n_neighbors': 22, 'model__weights': 'uniform', 'model__p': 2, 'model__leaf_size': 96, 'model__algorithm': 'kd_tree'}
-----------------------


In [33]:
df_results_knn

,sku,study,best_params,best_smape,best_gap,model,n_trials
0,SKU1,<optuna.study.study.Study object at 0x12fd5e450>,"{'model__n_neighbors': 2, 'model__weights': 'u...",44.00,5.90,KNeighborsRegressor,300
1,SKU10,<optuna.study.study.Study object at 0x139f95130>,"{'model__n_neighbors': 8, 'model__weights': 'u...",33.83,5.32,KNeighborsRegressor,300
2,SKU11,<optuna.study.study.Study object at 0x139f97d10>,"{'model__n_neighbors': 12, 'model__weights': '...",31.36,10.23,KNeighborsRegressor,300
3,SKU12,<optuna.study.study.Study object at 0x139f97890>,"{'model__n_neighbors': 23, 'model__weights': '...",80.13,25.90,KNeighborsRegressor,300
4,SKU13,<optuna.study.study.Study object at 0x139f96330>,"{'model__n_neighbors': 25, 'model__weights': '...",49.93,12.65,KNeighborsRegressor,300
5,SKU14,<optuna.study.study.Study object at 0x139f969f0>,"{'model__n_neighbors': 16, 'model__weights': '...",68.57,31.86,KNeighborsRegressor,300
6,SKU15,<optuna.study.study.Study object at 0x139f97c80>,"{'model__n_neighbors': 10, 'model__weights': '...",50.02,23.55,KNeighborsRegressor,300
7,SKU16,<optuna.study.study.Study object at 0x139f945f0>,"{'model__n_neighbors': 4, 'model__weights': 'u...",23.20,9.00,KNeighborsRegressor,300
8,SKU17,<optuna.study.study.Study object at 0x139f95be0>,"{'model__n_neighbors': 12, 'model__weights': '...",37.92,18.91,KNeighborsRegressor,300
9,SKU18,<optuna.study.study.Study object at 0x13b7c00e0>,"{'model__n_neighbors': 4, 'model__weights': 'u...",41.05,12.04,KNeighborsRegressor,300


In [34]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_knn[df_results_knn["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Selección mejor modelo y ajuste final

In [ ]:
# Eliminamos los estudios de Optuna de todos los DataFrames de resultados
# df_results_xgboost = df_results_xgboost.drop(columns=["study"])
# df_results_rf = df_results_rf.drop(columns=["study"])
# df_results_elastic = df_results_elastic.drop(columns=["study"])
# df_results_knn = df_results_knn.drop(columns=["study"])

# Union de resultados de los modelos
df_results = pd.concat(
    [df_results_xgboost, df_results_rf, df_results_elastic, df_results_knn],
    ignore_index=True,
).sort_values(by=["sku", "best_smape"], ascending=[True, True])

# Por cada SKU, obtenemos el mejor modelo
df_best_models = df_results.loc[
    df_results.groupby("sku")["best_smape"].idxmin()
].reset_index(drop=True)


# Ordenamos por mejor SMAPE
df_best_models = df_best_models.sort_values(by="best_smape", ascending=True).drop(
    columns=["study"]
)

In [36]:
# guardar en OUTPUT_DIR el dataframe de resultados por SKU
output_file = os.path.join(OUTPUT_DIR, "df_ML_models_results.xlsx")
df_results.to_excel(output_file, index=False)

df_results

,sku,study,best_params,best_smape,best_gap,model,n_trials
0,SKU1,<optuna.study.study.Study object at 0x1380c0ad0>,"{'max_depth': 2, 'learning_rate': 0.021, 'n_es...",27.46,1.83,XGBRegressor,125
52,SKU1,<optuna.study.study.Study object at 0x12fd5e8d0>,"{'model__alpha': 0.041858227295469716, 'model_...",27.69,23.54,ElasticNet,200
26,SKU1,<optuna.study.study.Study object at 0x105e7f140>,"{'n_estimators': 600, 'max_depth': 10, 'min_sa...",32.71,9.59,RandomForestRegressor,125
78,SKU1,<optuna.study.study.Study object at 0x12fd5e450>,"{'model__n_neighbors': 2, 'model__weights': 'u...",44.00,5.90,KNeighborsRegressor,300
1,SKU10,<optuna.study.study.Study object at 0x1380634d0>,"{'max_depth': 5, 'learning_rate': 0.076, 'n_es...",25.84,3.78,XGBRegressor,125
...,...,...,...,...,...,...,...
50,SKU8,<optuna.study.study.Study object at 0x139f94050>,"{'n_estimators': 1000, 'max_depth': 1, 'min_sa...",91.60,31.96,RandomForestRegressor,125
77,SKU9,<optuna.study.study.Study object at 0x139f970b0>,"{'model__alpha': 0.06032030579768781, 'model__...",50.18,8.58,ElasticNet,200
51,SKU9,<optuna.study.study.Study object at 0x139f94440>,"{'n_estimators': 1000, 'max_depth': 7, 'min_sa...",56.23,23.78,RandomForestRegressor,125
25,SKU9,<optuna.study.study.Study object at 0x12fd5ea80>,"{'max_depth': 2, 'learning_rate': 0.065, 'n_es...",59.79,41.27,XGBRegressor,125


In [44]:
# guardar en OUTPUT_DIR el dataframe de mejores modelos por SKU
output_file = os.path.join(OUTPUT_DIR, "df_ML_best_models.xlsx")
df_best_models.to_excel(output_file, index=False)

df_best_models

,sku,best_params,best_smape,best_gap,model,n_trials
21,SKU5,"{'max_depth': 3, 'learning_rate': 0.095, 'n_es...",16.14,11.97,XGBRegressor,125
20,SKU4,"{'n_estimators': 100, 'max_depth': 9, 'min_sam...",18.10,8.86,RandomForestRegressor,125
19,SKU3,"{'model__alpha': 0.000139345022513376, 'model_...",20.62,10.68,ElasticNet,200
7,SKU16,"{'model__alpha': 0.10558813779064824, 'model__...",20.88,14.66,ElasticNet,200
11,SKU2,"{'max_depth': 5, 'learning_rate': 0.02, 'n_est...",24.81,12.56,XGBRegressor,125
15,SKU23,"{'n_estimators': 250, 'max_depth': 7, 'min_sam...",24.83,11.49,RandomForestRegressor,125
22,SKU6,"{'model__alpha': 0.00011293569666047012, 'mode...",25.80,4.81,ElasticNet,200
1,SKU10,"{'max_depth': 5, 'learning_rate': 0.076, 'n_es...",25.84,3.78,XGBRegressor,125
8,SKU17,"{'model__alpha': 0.0012363188277052211, 'model...",26.54,21.25,ElasticNet,200
18,SKU26,"{'model__alpha': 0.10558813779064824, 'model__...",27.46,16.67,ElasticNet,200


# Predicción y graficas 

In [45]:
# Cargar tabla con mejor modelo por SKU
df_best_models_2 = pd.read_excel(
    os.path.join(OUTPUT_DIR, "df_ML_best_models.xlsx"),
    engine="openpyxl",
)

In [50]:
df_best_models_2.sample(2)

,sku,best_params,best_smape,best_gap,model,n_trials
8,SKU17,"{'model__alpha': 0.0012363188277052211, 'model...",26.54,21.25,ElasticNet,200
2,SKU3,"{'model__alpha': 0.000139345022513376, 'model_...",20.62,10.68,ElasticNet,200


In [38]:
# # Reentrenamos la serie de tiempo de cada SKU con el mejor modelo, usando el conjunto de entrenamiento completo
# for sku in df_best_models["sku"].unique():
#     print(f"\n🔍 Reentrenando modelo para SKU: {sku}\n")

#     # Filtramos el DataFrame para el SKU actual
#     df_sku = df[df["sku"] == sku].copy()

#     # Definimos las variables predictoras
#     X = df_sku[features]
#     y = df_sku["pedidos"]

#     # Obtenemos el mejor modelo y sus parámetros
#     best_model_name = df_best_models.loc[df_best_models["sku"] == sku, "model"].values[
#         0
#     ]
#     best_params = df_best_models.loc[
#         df_best_models["sku"] == sku, "best_params"
#     ].values[0]

#     # Creamos el modelo con los mejores parámetros
#     if best_model_name == "XGBRegressor":
#         model = XGBRegressor(**best_params)
#     elif best_model_name == "RandomForestRegressor":
#         model = RandomForestRegressor(**best_params)
#     elif best_model_name == "ElasticNet":
#         model = Pipeline(
#             [
#                 ("scaler", StandardScaler()),
#                 ("model", ElasticNet(**best_params)),
#             ]
#         )
#     else:
#         raise ValueError(f"Modelo desconocido: {best_model_name}")

#     # Entrenamos el modelo con todo el conjunto de entrenamiento
#     model.fit(X, y)

#     # Guardamos el modelo entrenado en un archivo
#     ml_utils.save_model(model, sku, OUTPUT_DIR)
